In [ ]:
# prompt: mount to drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!pip install transformers tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast
import os
import glob

In [ ]:
MERGED_TOKENIZER_DIR = "/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL"

In [ ]:
# === Step 1: Load the two tokenizers ===
old = Tokenizer.from_file("/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2_MERGED/tokenizer.json")
new = Tokenizer.from_file("/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf/tokenizer.json")

In [ ]:
OLD_TOKENIZER_PATH = "/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2"
NEW_TOKENIZER_PATH = "/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf"

In [ ]:
MERGED_TOKENIZER_PATH = "/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL"

In [ ]:
from transformers import PreTrainedTokenizerFast

# Load both
old_tokenizer = PreTrainedTokenizerFast.from_pretrained(OLD_TOKENIZER_PATH)
new_tokenizer = PreTrainedTokenizerFast.from_pretrained(NEW_TOKENIZER_PATH)

# Get new tokens only
old_tokens = set(old_tokenizer.get_vocab().keys())
new_tokens = set(new_tokenizer.get_vocab().keys())

tokens_to_add = [tok for tok in new_tokens if tok not in old_tokens and isinstance(tok, str)]

print(f"🔍 Tokens to add: {len(tokens_to_add)}")

# Actually add them
num_added = old_tokenizer.add_tokens(tokens_to_add, special_tokens=False)
print(f"✅ Tokens actually added: {num_added}")

# Save merged
old_tokenizer.save_pretrained(MERGED_TOKENIZER_PATH)

🔍 Tokens to add: 12778
✅ Tokens actually added: 12778


('/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL/tokenizer_config.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL/special_tokens_map.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL/tokenizer.json')

In [ ]:
print("🔎 New vocab size:", len(old_tokenizer.get_vocab()))

🔎 New vocab size: 107725


In [ ]:
prompt = "ما معنى الحرية؟"
tokens = old_tokenizer.tokenize(prompt)
print("🔤 Tokens:", tokens)
print("🧬 IDs:", old_tokenizer.convert_tokens_to_ids(tokens))

🔤 Tokens: ['▁ما', '▁معنى', '▁الحرية', '<0xD8>', '<0x9F>']
🧬 IDs: [0, 0, 0, 219, 162]


In [ ]:
from transformers import PreTrainedTokenizerFast

# ✅ Path to your merged tokenizer
MERGED_TOKENIZER_PATH = "/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL"

# ✅ Load tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained(MERGED_TOKENIZER_PATH)

# === 🧪 Sample prompts for testing ===
samples = [
    "ما هو الذكاء الاصطناعي؟",                                 # formal Arabic
    "مَا هُوَ الذَّكَاءُ الِاصْطِنَاعِيُّ؟",                   # fully diacritized
    "ازاي أشتغل على موديل LLaMA بالعامية؟",                   # Egyptian dialect + code term
    "كان يا ما كان في قديم الزمان، كان هناك أميرٌ شجاعٌ."       # storytelling with light diacritics
]

# === 🔎 Run tests ===
for sample in samples:
    print("🔤 Original:", sample)
    token_ids = tokenizer.encode(sample)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    print("🧩 Tokens:", tokens)
    print("🧱 Token IDs:", token_ids)
    print("🔁 Decoded:", tokenizer.decode(token_ids))
    print(f"❗ [UNK] count: {tokens.count('[UNK]')}")
    print("—" * 60)

🔤 Original: ما هو الذكاء الاصطناعي؟
🧩 Tokens: ['<s>', 'ما', 'هو', 'الذكاء', 'الاصطناعي', '<0xD8>', '<0x9F>']
🧱 Token IDs: [1, 38629, 37451, 35541, 99831, 219, 162]
🔁 Decoded: <s>ماهوالذكاءالاصطناعي؟
❗ [UNK] count: 0
————————————————————————————————————————————————————————————
🔤 Original: مَا هُوَ الذَّكَاءُ الِاصْطِنَاعِيُّ؟
🧩 Tokens: ['<s>', 'مَا', 'هُوَ', 'الذ', 'َ', 'ّ', 'ك', 'َ', 'ا', 'ء', 'ُ', 'الِاصْ', 'ط', 'ِ', 'ن', 'َ', 'ا', 'ع', 'ِ', 'ي', 'ُ', 'ّ', '<0xD8>', '<0x9F>']
🧱 Token IDs: [1, 86364, 80210, 48018, 30323, 30857, 30283, 30323, 30112, 30992, 30711, 66442, 30367, 30567, 30162, 30323, 30112, 30218, 30567, 30163, 30711, 30857, 219, 162]
🔁 Decoded: <s>مَاهُوَالذَّكَاءُالِاصْطِنَاعِيُّ؟
❗ [UNK] count: 0
————————————————————————————————————————————————————————————
🔤 Original: ازاي أشتغل على موديل LLaMA بالعامية؟
🧩 Tokens: ['<s>', 'ازاي', 'أش', 'ت', 'غ', 'ل', 'على', 'مودي', 'ل', '▁L', 'La', 'MA', 'بالعام', 'ي', 'ة', '<0xD8>', '<0x9F>']
🧱 Token IDs: [1, 33358, 55764, 30195, 30611

In [ ]:
from transformers import PreTrainedTokenizerFast
import os

# Paths
MERGED_TOKENIZER_DIR = "/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL"
TOKENIZER_JSON_PATH = os.path.join(MERGED_TOKENIZER_DIR, "tokenizer.json")

# Load tokenizer
tokenizer = PreTrainedTokenizerFast(tokenizer_file=TOKENIZER_JSON_PATH)

# Show vocab size
print(f"✅ Final merged vocab size: {tokenizer.vocab_size}")

# Optionally, print some sample tokens
print("🧪 Sample tokens from merged tokenizer:")
print(tokenizer.convert_ids_to_tokens(list(range(100))))

✅ Final merged vocab size: 32000
🧪 Sample tokens from merged tokenizer:
['<unk>', '<s>', '</s>', '<0x00>', '<0x01>', '<0x02>', '<0x03>', '<0x04>', '<0x05>', '<0x06>', '<0x07>', '<0x08>', '<0x09>', '<0x0A>', '<0x0B>', '<0x0C>', '<0x0D>', '<0x0E>', '<0x0F>', '<0x10>', '<0x11>', '<0x12>', '<0x13>', '<0x14>', '<0x15>', '<0x16>', '<0x17>', '<0x18>', '<0x19>', '<0x1A>', '<0x1B>', '<0x1C>', '<0x1D>', '<0x1E>', '<0x1F>', '<0x20>', '<0x21>', '<0x22>', '<0x23>', '<0x24>', '<0x25>', '<0x26>', '<0x27>', '<0x28>', '<0x29>', '<0x2A>', '<0x2B>', '<0x2C>', '<0x2D>', '<0x2E>', '<0x2F>', '<0x30>', '<0x31>', '<0x32>', '<0x33>', '<0x34>', '<0x35>', '<0x36>', '<0x37>', '<0x38>', '<0x39>', '<0x3A>', '<0x3B>', '<0x3C>', '<0x3D>', '<0x3E>', '<0x3F>', '<0x40>', '<0x41>', '<0x42>', '<0x43>', '<0x44>', '<0x45>', '<0x46>', '<0x47>', '<0x48>', '<0x49>', '<0x4A>', '<0x4B>', '<0x4C>', '<0x4D>', '<0x4E>', '<0x4F>', '<0x50>', '<0x51>', '<0x52>', '<0x53>', '<0x54>', '<0x55>', '<0x56>', '<0x57>', '<0x58>', '<0x59>', '<0

In [ ]:
# Print the last 100 tokens in the vocab
print("\n🧪 Last 100 tokens in the merged tokenizer:")
print(tokenizer.convert_ids_to_tokens(list(range(tokenizer.vocab_size - 100, tokenizer.vocab_size))))


🧪 Last 100 tokens in the merged tokenizer:
['书', '构', '米', '连', '操', '装', '과', 'ぐ', '反', '̌', '仮', '员', '昭', 'ശ', '兴', '客', '删', 'ම', 'ව', 'პ', 'ċ', 'ഷ', 'သ', 'ᵉ', '居', '타', '𝓝', 'थ', '現', 'ˇ', '종', '助', '唐', '瀬', 'ន', '微', '１', 'Ġ', 'ほ', '舞', '내', '중', 'Ē', '导', '效', '방', 'ḏ', '深', '梅', '料', '월', '每', '洲', '회', '茶', '败', 'ഞ', 'ể', 'ヨ', '些', '双', '嘉', '모', '바', 'ษ', '進', '음', 'ญ', '丁', '故', '計', '遠', '교', '재', '候', '房', '명', '两', 'ფ', '才', '합', '止', '番', 'ɯ', '奇', '怪', '联', '역', '泰', '백', 'ὀ', 'げ', 'べ', '边', '还', '黃', '왕', '收', '弘', '给']


In [ ]:
from tokenizers import Tokenizer

# Load raw tokenizers
old_raw = Tokenizer.from_file(os.path.join(OLD_TOKENIZER_PATH, "tokenizer.json"))
new_raw = Tokenizer.from_file(os.path.join(NEW_TOKENIZER_PATH, "tokenizer.json"))

# Compare tokens
original_tokens = set(old_raw.get_vocab().keys())
new_tokens = set(new_raw.get_vocab().keys())
added = [tok for tok in new_tokens if tok not in original_tokens]

print(f"🧠 Actually added: {len(added)}")
print("🧾 Sample new tokens added:", added[:20])

🧠 Actually added: 12778
🧾 Sample new tokens added: ['وعرضه', 'المكتسبة', 'الهيدروجين', 'بالمواضيع', 'لنمذجة', 'الأكسدة', 'استئصال', 'ومستويات', 'بالألوان', 'وحوا', 'جدولة', 'تصفي', 'إشراكًا', 'اللامرك', 'الإشعاعي', 'مقسومة', 'أيرلندا', 'كانَ', 'التحليلات', 'الاستعلام']


In [ ]:
from transformers import PreTrainedTokenizerFast

# Load your merged tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained("/content/drive/MyDrive/FINAL_GRAD_PROJ/TOKENIZER_FINETUNED_FINAL")

# Print vocab size
print(f"🔢 Total vocab size: {len(tokenizer.get_vocab())}")

🔢 Total vocab size: 107725


In [ ]:
prompt = "ازاي أقدر أكتب كود لذكاء اصطناعي بيتكلم مصري؟"
tokens = tokenizer.tokenize(prompt)
print("🧩 Tokens:", tokens)
print(f"❗ [UNK] count: {tokens.count('[UNK]')}")

🧩 Tokens: ['▁ازاي', '▁أقدر', '▁أكتب', '▁كود', '▁لذ', 'ك', 'ا', 'ء', '▁اصطناعي', '▁بيتك', 'ل', 'م', '▁مصري', '<0xD8>', '<0x9F>']
❗ [UNK] count: 0
